# Zirconia classification tutorial

In [ ]:
import math
import sys
from pathlib import Path

import torch
import torchmetrics
from lightning import Trainer, seed_everything
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from torch.nn import Embedding, Sequential

sys.path.append("../..")

import pandas as pd
from matplotlib import pyplot as plt
from scripts.confusion_matrix import run_confusion_matrix
from src import LightningDataset, Module
from src.constants import DEFAULT_SEED
from src.models.expansion.radial import RadialBesselBasis
from src.models.painn import PaiNN, PaiNNRadial
from src.transforms import BoxStrain, RandomPerturbation
from src.typing import PathLike

In [ ]:
# Main variables that can be changed to optimize training
EPOCHS: int = 1_000  # number of epochs during training
NUM_NEIGHBORS: int = 12  # number of neighbors in graphs
BATCH_SIZE: int = 2048  # batch size during training (changed from 512 to 32 for memory)

INFERENCE_BATCH_SIZE: int = 4096

COMPILE: bool = torch.cuda.is_available()  # set True of CUDA, False otherwise

# To read a full checkpoint to re-train:
CKPT_NAME: PathLike | None = "asc_checkpoint.ckpt"  # To re-read a checkpoint

# To only read weights for inference:
WEIGHTS_NAME: PathLike | None = "asc_weights.pt"  # To re-read a checkpoint

# Configuration file to test the inference
TO_PREDICT: list[PathLike] = ["configs_zro2/tetra_tZ_5_5_5.lmp"]  # .lmp file to test the model

In [ ]:
# To whitelist Python classes/functions so that PyTorch is allowed to unpickle them
# when loading a checkpoint
torch.serialization.add_safe_globals(
    [
        PaiNN,
        Embedding,
        Sequential,
        PaiNNRadial,
        RadialBesselBasis,
    ]
)

torch.set_float32_matmul_precision(
    "medium"
)  # allows PyTorch to use faster slightly less precise algorithms especially on GPU (Tensor Cores)

torch.backends.cudnn.benchmark = True  # optimizes convolution performance (CUDA only)
_ = seed_everything(DEFAULT_SEED)  # reproducible random seed


def pick_precision() -> str | int:
    """Picks the best precision for the current GPU.

    Returns:
        str | int: The chosen precision.
    """
    if not torch.cuda.is_available():
        return 32
    major, _ = torch.cuda.get_device_capability()
    return "bf16-mixed" if major >= 8 else "16-mixed"

To mitigate overfitting and improve generalization, we can apply data augmentations to the training data. Here, we use three augmentations designed for atomistic data:
1. RandomPerturbation: Adds gaussian noise to atomic positions, emulating thermal vibrations (here width of Gaussian is 0.05 Angstrom).
3. BoxStrain: Adds random strains to the simulation box, emulating various types of deformations (here width of Gaussian is 5%).

Since the provided silicon dataset is quite imbalanced, we enable imbalance sampling to over-sample the less represented classes. We also split the dataset in 3 parts. This is crucial to ensure a model generalizes well to new, unseen data. This "hold-out" strategy prevents the model from simply memorizing (overfitting) the training data and is commonly used in supervised machine-learning. Here, we use 70% of the dataset for training, 20% for validation, and 10% for testing.

In [ ]:
# augmentation: add random displacements and strains to the data set to improve training
augmentations = [
    RandomPerturbation(std_range=(0.0, 0.05)),
    BoxStrain(
        std_range=(0.0, 0.05),
        directions="all",
    ),
]

# Here will use 70% for training, 20% for validation, 10% for testing
datamodule = LightningDataset(
    dataset_name="custom",
    lengths=(0.7, 0.2, 0.1),
    root="data",
    transforms=augmentations,
    k=NUM_NEIGHBORS,
    num_workers=8,
    persistent_workers=True,
    batch_size=BATCH_SIZE,
    use_imbalance_sampler=True,
    # force_reload=True,  # force reload of the dataset (useful if augmentations have changed)
    # uncomment if deadlocks appear with num_workers > 0 and after graph computation
    multiprocessing_context="spawn",
)

In PyTorch Lightning, the Trainer class acts as the "project manager", connecting the model and data logic to the underlying hardware (CPU, GPU, ...) while handling background tasks like checkpointing and validation. It is also used to run the training loop, saving checkpoints, logging...

In this example, we want the trainer to:
1. Print a summary of the model architecture.
2. Find the optimal batch size before training starts (will override the batch size defined above!).
3. Log the training metrics in a CSV file and with Tensorboard.
4. Save the best checkpoint based on the minimum validation loss.
5. Use mixed precision if a GPU is available for faster training and reduced memory usage.
6. Enable the progress bar to monitor training progress.

The settings can be adjusted based on your specific needs and preferences. You can refer to the PyTorch Lightning documentation for more details on available callbacks and trainer options: https://pytorch-lightning.readthedocs.io/en/stable/api/pytorch_lightning.Trainer.html

Note that changing the batch size will also change the optimal learning rate, which is NOT adjusted accordingly in this example (we are using a learning rate adapted for a batch size of 512). You can try to disable this callback to see how it affects learning.

In [ ]:
callbacks = [
    # BatchSizeFinder(init_val=128, max_trials=5, mode="power"),
    # DR I commented the following line
    # HalfBatchSizeFinder(init_val=128, max_trials=5),
    ModelCheckpoint(
        dirpath="./checkpoints",
        monitor="val/loss",
        mode="min",
        every_n_epochs=1,
        save_weights_only=True,
        save_top_k=5,  # Save the top 5 models based on validation loss
        auto_insert_metric_name=False,
        filename="painn-epoch={epoch}-val_loss={val/loss:.4f}",
    ),
]

loggers = [
    CSVLogger(save_dir="../../logs", name="painn_experiment"),
    TensorBoardLogger("../../logs", name="painn_experiment"),
]

# NOTE: MPS (Apple Silicon GPU) is disabled here because some PyTorch Geometric
# operations are not yet fully supported on MPS. Switch to accelerator='gpu'
# and remove devices=1 override once MPS support improves, or when running on CUDA.
trainer = Trainer(
    max_epochs=EPOCHS,
    precision=pick_precision(),
    callbacks=callbacks,
    logger=loggers,
    enable_progress_bar=True,
    enable_model_summary=True,
    log_every_n_steps=1,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
)

In this final initialization stage, we prepare the model before training. Optionally, we can load a pre-trained checkpoint to continue training or perform inference on new data. Let's break down the different part of the code.

First, we define the different metrics we will monitor during training. Here, we chose to check the accuracy (fraction of correct predictions), the F1 score (the mean of the precision (exactness) and recall (completeness)), and AUROC (measure the ability of the model to distinguish between classes). Note that the loss is not included as a metric but will be computed anyway since it is necessary for training a neural network. The last metric we monitor is the confusion matrix, which reports the number of true positives, false negatives, false positives, and true negatives. This allows a more detailed analysis than simply observing the metrics defined above, especially if the dataset is unbalanced. Being a matrix, this metric will not be logged in the CSV file but with Tensorboard only.

Then, we check if a checkpoint was the provided. If not, we initialize a new model. The parameters provided here allow to reach close to perfect accuracy on this dataset with minimum computational time. Feel free to play with the different parameters to understand how they influence the results.

While multiple model architectures are provided, we use a modified PaiNN architecture (Schütt et al., International Conference on Machine Learning (2021)) that allows to use vectorial features (i.e, direction vectors) in an equivariant manner, resulting in cheap and accurate models.

In [ ]:
num_classes = datamodule.num_classes
metrics = [
    torchmetrics.Accuracy(task="multiclass", num_classes=num_classes),
    torchmetrics.F1Score(task="multiclass", num_classes=num_classes),
    torchmetrics.AUROC(task="multiclass", num_classes=num_classes),
    torchmetrics.ConfusionMatrix(task="multiclass", num_classes=num_classes),
]

datamodule.setup("fit")

steps_per_epoch = math.ceil(len(datamodule.train_dataset) / BATCH_SIZE)

should_train = CKPT_NAME is None or not Path(CKPT_NAME).exists()
read_weights = WEIGHTS_NAME is not None and Path(WEIGHTS_NAME).exists()

with trainer.init_module():
    model = Module(
        model=PaiNN(
            out_channels=num_classes,
            num_radial=8,
            hidden_channels=32,
            num_layers=4,  # not ok k={10, 14, 16, 20}, ok k={12, 18}
            dropout=0.5,
            scale_factor=1.0 / math.sqrt(NUM_NEIGHBORS),
        ),
        metrics=metrics,
        compile=COMPILE,
        warmup=400,
        lr=0.004678965862088063,
        max_iters=EPOCHS * steps_per_epoch,
    )

# DR: Made a change here to only read weights
if read_weights:
    print(f"Loading weights from: {WEIGHTS_NAME}")
    state_dict = torch.load(WEIGHTS_NAME, map_location="cpu")
    model.load_state_dict(state_dict)

We can now train the model! After training, we use the best model checkpoint to perform 1 epoch on the validation and test sets to ensure the model generalize well to unseen data.

In [ ]:
if not read_weights:
    if should_train:
        print("Training from scratch")
        trainer.fit(model=model, datamodule=datamodule, weights_only=False)
    else:
        print(f"Resuming training from {CKPT_NAME}")

        ckpt = torch.load(CKPT_NAME, map_location="cpu", weights_only=False)
        print("epoch =", ckpt["epoch"])

        trainer.fit_loop.max_epochs = ckpt["epoch"] + EPOCHS

        trainer.fit(model=model, datamodule=datamodule, ckpt_path=CKPT_NAME, weights_only=False)

    trainer.validate(datamodule=datamodule, weights_only=False)
    trainer.test(datamodule=datamodule, weights_only=False)

    # To save the full checkpoint (to restart training) and lightweight weights (for inference)
    trainer.save_checkpoint("asc_checkpoint.ckpt")
    torch.save(model.state_dict(), "asc_weights.pt")

Let's see if our model learned correctly. For that, we can plot the different metrics we computed during training and validation. Ideally, the accuracy, F1 score and AUROC must converge to 1, while the loss must go to 0. The best model is usually the one that yields the best validation metrics. In our case, you might remember that we defined our best model as the one with the lowest validation loss.

In [ ]:
def plot_metrics(trainer: Trainer) -> None:
    """Plot training and validation curves for accuracy, F1 score, AUROC, and loss.

    Args:
        trainer: The PyTorch Lightning Trainer object after training is complete.
    """
    if not (trainer.logger and trainer.logger.log_dir):
        print(
            "No logger or log directory, skipping training curves visualization. "
            "Make sure to use a logger that saves metrics to a file (e.g., CSVLogger)."
        )
        return

    metric_path = Path(trainer.logger.log_dir) / "metrics.csv"
    if not metric_path.exists():
        print(f"No metrics file found at {metric_path}, skipping training curves visualization.")
        return

    df = pd.read_csv(metric_path)

    # We are logging every n steps during training, so we average
    # the metrics per epoch to get a single value per epoch
    df = df.groupby("epoch").mean(numeric_only=True)

    # Define the metric pairs we want to visualize
    metric_groups = [
        ("Accuracy (↑)", "train/MulticlassAccuracy", "val/MulticlassAccuracy"),
        ("F1 Score (↑)", "train/MulticlassF1Score", "val/MulticlassF1Score"),
        ("AUROC (↑)", "train/MulticlassAUROC", "val/MulticlassAUROC"),
        ("Loss (↓)", "train/loss", "val/loss"),
    ]

    # Filter out metric groups where neither training nor validation metric is available
    available_plots = [p for p in metric_groups if p[1] in df.columns or p[2] in df.columns]

    _, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)

    for ax, (title, t_col, v_col) in zip(axes.flat, available_plots):
        # Train metric
        if t_col in df.columns:
            train_data = df[t_col].dropna()
            ax.plot(train_data.index, train_data, label="Train", linestyle="-", marker="o")

        # Validation metric
        if v_col in df.columns:
            val_data = df[v_col].dropna()
            ax.plot(val_data.index, val_data, label="Val", linestyle="--", marker="x")

        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


if not read_weights:
    plot_metrics(trainer)
else:
    print("Model was loaded from weights, skipping training curves visualization.")

For a more in-depth analysis of the performance of the trained model, we can also visualize the confusion matrix. This will allow us to see how well the model is performing on each class and identify any potential issues with class imbalance or misclassification. For this, we can use the `run_confusion_matrix` function, which will compute and plot the confusion matrix for the specified data split (train, validation, test, or all), along with other useful metrics such as precision, recall, and F1 score. The results are both displayed in the notebook and saved to the specified output directory `out_dir` for further analysis if needed.

Please note that, in the case where there are many classes, the confusion matrix can become quite large and difficult to interpret. In such cases, it may be helpful to use the `compact_matrix_plot` option to optimize the rendering of the confusion matrix by suppressing cell text and reducing axis tick density. This can make it easier to visualize the overall performance of the model across all classes.

In [ ]:
checkpoint_path = Path("./checkpoints/painn-epoch=520-val_loss=0.0389.ckpt")

if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

datamodule_kwargs = {
    "root": "data",
    "multiprocessing_context": "spawn",
    "lengths": (0.7, 0.2, 0.1),
}

loader_kwargs = {
    "weights_only": False,
}

out_dir = "evaluation_results_best_checkpoint"

for split in ("train", "val", "test", "all"):
    print(f"Running confusion-matrix for split={split}")
    run_confusion_matrix(
        checkpoint=str(checkpoint_path),
        dataset=datamodule.dataset_name,
        split=split,
        num_neighbors=NUM_NEIGHBORS,
        batch_size=INFERENCE_BATCH_SIZE,
        num_workers=8,
        output_dir=out_dir,
        normalize=True,
        save_predictions=True,
        compact_matrix_plot=False,
        datamodule_kwargs=datamodule_kwargs,
        loader_kwargs=loader_kwargs,
    )

print("\nDone!")

Regardless of the split we chose (train, validation, test, or all the data) we see that the model performs very well on all classes. In the case of $ZrO_2$, we only notice minor confusion (a few percentage points at most) between classes 3 and 6, as shown by the almost imperceptible non-white off-diagonal elements in the plots. Such visual information can be critical to understand the model's behavior. For instance, in this specific case one could look at the atomic structures of classes 3 and 6 to see whether they are similar and thus more difficult to distinguish (especially at non-zero temperatures) or if one could legitimately expect an improvement in the model's performance.

In this final section of this tutorial, we move from training to inference. We will see how we can run our trained model on very large systems and how we can save the results to a format compatible with visualization tools such as OVITO. Note that we also provide an OVITO Python modifier that you can download and run directly from the OVITO software!

To enable fast inference, we make use of PyTorch Geometric's NeighborLoader class that allow us quickly process large graphs while avoiding the neighborhood explosion when the number of layers increase. In a message-passing neural network like used here, each layer expands the reach of a single atom. For a model with *L* layers and *k* neighbors, we will have to compute the embeddings of $\sum\limits_{i=0}^{L} k^i$ atoms for each central atom! When using hierarchical neighborhood sampling, we can significantly reduce the number of computations required by progressively "trimming" the graph as it passes through each layer. The core insight behind this technique is that nodes sampled in the later "hops" of the neighborhood are only necessary for the initial message-passing layers. As the data flows deeper into the GNN, these distant nodes no longer contribute to the final representation of the central seed atom. This allow us to progressively trim the graph as it passes through each layer.

In [ ]:
%%script false --no-raise-error

import numpy as np
from ovito.data import DataCollection
from ovito.io import export_file, import_file
from torch import Tensor
import json
import tqdm
from src.graph import PeriodicKNN
from torch_geometric.loader import NeighborLoader

# to avoid unnecessary warnings
import warnings
warnings.filterwarnings("ignore", message=".*OVITO.*PyPI")

def load_index_to_label(mapping_path: str | Path) -> dict[int, int]:
    """Loads a JSON file that maps model class indices to space group numbers.

    Args:
        mapping_path (str | Path): Path to the JSON file containing the mapping.

    Returns:
        dict[int, int]: Dictionary mapping model class indices (int) to space group numbers (int).
    """
    with open(mapping_path) as f:
        mapping = json.load(f)
    return {int(k): int(v) for k, v in mapping.items()}


@torch.inference_mode()
def inference(model: Module, data: DataCollection) -> tuple[Tensor, Tensor]:
    """Run inference on a single DataCollection object.

    Returns per-atom predictions and a single majority-vote prediction for the
    whole structure. The majority vote aggregates all per-atom class indices and
    picks the most frequent one, giving a single space-group prediction per
    configuration regardless of atomic species.

    Args:
        model: The trained PyTorch Lightning Module for prediction.
        data: The input DataCollection object containing the particle data to predict on.

    Returns:
        per_atom_preds: Per-atom predicted class indices (shape [num_atoms]).
        structure_pred: Single majority-vote prediction for the structure (scalar tensor).
    """
    model.eval()
    device = next(model.parameters()).device
    num_layers: int = model.model.num_layers  # type: ignore

    knn = PeriodicKNN(k=NUM_NEIGHBORS)
    graph = knn.convert(data)

    print(f"Graph has {graph.num_nodes} nodes and {graph.num_edges} edges.")

    loader = NeighborLoader(
        graph,
        num_neighbors=[-1] * num_layers,
        batch_size=min(INFERENCE_BATCH_SIZE, graph.num_nodes),  # type: ignore
        shuffle=False,
        num_workers=0,
        persistent_workers=False,
        pin_memory=(device.type == "cuda"),
    )

    print(f"Running inference on {len(loader)} batches of size {loader.batch_size}...")

    graph_preds = []
    for batch in tqdm(loader, unit="batch", total=len(loader)):
        batch = batch.to(device)
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            out = model.predict_step(batch)
        graph_preds.append(out.to("cpu", non_blocking=(device.type == "cuda")))

    if device.type == "cuda":
        torch.cuda.synchronize()

    per_atom_preds = torch.cat(graph_preds, dim=0)

    # Majority vote: aggregate per-atom predictions into a single structure-level label.
    # torch.mode returns the most frequent value; [0] extracts the values tensor.
    structure_pred = torch.mode(per_atom_preds).values

    return per_atom_preds, structure_pred


def dump_outputs(
    per_atom_preds: Tensor,
    structure_pred: Tensor,
    data: DataCollection,
    fpath: PathLike,
    index_to_label: dict,
) -> None:
    """Save per-atom predictions and print the majority-vote structure prediction.

    Args:
        per_atom_preds: Per-atom predicted class indices (shape [num_atoms]).
        structure_pred: Single majority-vote prediction for the structure (scalar tensor).
        data: The OVITO DataCollection object to attach predictions to.
        fpath: Path to the input .lmp file (used to derive the output filename).
        index_to_label: Mapping from model class index to space group number.
    """
    path = Path(fpath)
    out_path = path.with_name(f"{path.stem}_predicted.extxyz")

    # Decode per-atom indices → space group numbers
    pred_array = per_atom_preds.detach().cpu().view(-1).numpy()
    pred_sg = np.array([index_to_label[int(i)] for i in pred_array])

    # Attach per-atom predictions to OVITO data for visualization
    data.particles_.create_property("SpaceGroupPrediction", data=pred_sg)

    export_file(
        data,
        str(out_path),
        "xyz",
        columns=[
            "Particle Identifier",
            "Particle Type",
            "Position.X",
            "Position.Y",
            "Position.Z",
            "SpaceGroupPrediction",
        ],
    )

    # Decode and print the majority-vote structure-level prediction
    sg_number = index_to_label[int(structure_pred.item())]
    print(f"Structure prediction (majority vote): space group #{sg_number}")
    print(f"Per-atom predictions saved to {out_path}")


processed_dir = Path(datamodule.dataset.processed_dir)
MAPPING_PATH = processed_dir / "index_to_label.json"

if not MAPPING_PATH.exists():
    raise FileNotFoundError(
        f"Mapping not found at {MAPPING_PATH}. "
        "Did you run dataset.process() after adding JSON export?"
    )

index_to_label = load_index_to_label(MAPPING_PATH)


if TO_PREDICT:
    print(f"Running inference on {len(TO_PREDICT)} files...")
    for path in tqdm(TO_PREDICT):
        data = import_file(path).compute()
        per_atom_preds, structure_pred = inference(model, data)
        dump_outputs(per_atom_preds, structure_pred, data, path, index_to_label)
else:
    print("No files specified for prediction.")